# Hospital Readmission Risk Modeling

This notebook builds a classification model to predict hospital risk buckets based on social vulnerability indicators.

In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, '..')

from src.utils.io import load_analytics

## Step 1 — Load Data

In [2]:
df = load_analytics()
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Loaded: 5,421 rows x 66 columns


,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Hospital Type,Hospital Ownership,...,EP_UNEMP,EP_NOHSDP,EP_UNINSUR,EP_NOVEH,EP_AGE65,EP_DISABL,err,high_err_flag,high_svi_flag,risk_bucket
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Acute Care Hospitals,Government - Hospital District or Authority,...,5.0,14.5,13.2,6.2,17.8,18.0,0.971467,0,1,High SVI + Low ERR
1,010005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,MARSHALL,(256) 593-8310,Acute Care Hospitals,Government - Hospital District or Authority,...,1.0,15.1,11.1,7.2,16.9,15.2,0.890575,0,1,High SVI + Low ERR
2,010006,NORTH ALABAMA MEDICAL CENTER,1701 VETERANS DRIVE,FLORENCE,AL,35630,LAUDERDALE,(256) 768-8400,Acute Care Hospitals,Proprietary,...,6.3,13.5,14.2,11.6,18.0,16.7,0.992550,0,1,High SVI + Low ERR
3,010007,MIZELL MEMORIAL HOSPITAL,702 N MAIN ST,OPP,AL,36467,COVINGTON,(334) 493-3541,Acute Care Hospitals,Voluntary non-profit - Private,...,5.0,14.8,12.5,7.6,24.6,22.6,1.028350,1,1,High SVI + High ERR
4,010008,CRENSHAW COMMUNITY HOSPITAL,101 HOSPITAL CIRCLE,LUVERNE,AL,36049,CRENSHAW,(334) 335-3374,Acute Care Hospitals,Proprietary,...,3.2,18.1,8.2,7.4,23.1,16.2,NaN,0,1,High SVI + Low ERR


In [3]:
# Check risk bucket distribution
df['risk_bucket'].value_counts(normalize=True)

risk_bucket
High SVI + Low ERR     0.370227
Low SVI + Low ERR      0.369673
High SVI + High ERR    0.153662
Low SVI + High ERR     0.106438
Name: proportion, dtype: float64

## Step 2 — Create Modeling Dataset

Select only meaningful columns for modeling.

In [4]:
features = [
    'EP_POV150',   # % below 150% poverty
    'EP_UNEMP',    # % unemployed
    'EP_NOHSDP',   # % no high school diploma
    'EP_UNINSUR',  # % uninsured
    'EP_AGE65',    # % age 65+
    'EP_DISABL',   # % with disability
    'EP_NOVEH'     # % no vehicle
]

target = 'risk_bucket'

model_df = df[features + [target]].copy()
print(f"Model dataset: {model_df.shape}")
model_df.head()

Model dataset: (5421, 8)


,EP_POV150,EP_UNEMP,EP_NOHSDP,EP_UNINSUR,EP_AGE65,EP_DISABL,EP_NOVEH,risk_bucket
0,32.0,5.0,14.5,13.2,17.8,18.0,6.2,High SVI + Low ERR
1,32.3,1.0,15.1,11.1,16.9,15.2,7.2,High SVI + Low ERR
2,37.7,6.3,13.5,14.2,18.0,16.7,11.6,High SVI + Low ERR
3,30.1,5.0,14.8,12.5,24.6,22.6,7.6,High SVI + High ERR
4,22.1,3.2,18.1,8.2,23.1,16.2,7.4,High SVI + Low ERR


In [5]:
# Check for missing values
print("Missing values:")
print(model_df.isnull().sum())
print(f"\nTotal rows with any missing: {model_df.isnull().any(axis=1).sum()}")

Missing values:
EP_POV150      227
EP_UNEMP       219
EP_NOHSDP      200
EP_UNINSUR     223
EP_AGE65       199
EP_DISABL      223
EP_NOVEH       235
risk_bucket      0
dtype: int64

Total rows with any missing: 237


In [6]:
# Drop rows with missing values for modeling
model_df_clean = model_df.dropna()
print(f"Clean dataset: {model_df_clean.shape[0]:,} rows ({100*len(model_df_clean)/len(model_df):.1f}% of original)")

Clean dataset: 5,184 rows (95.6% of original)


In [7]:
# Feature summary statistics
model_df_clean[features].describe()

,EP_POV150,EP_UNEMP,EP_NOHSDP,EP_UNINSUR,EP_AGE65,EP_DISABL,EP_NOVEH
count,5184.000000,5184.000000,5184.000000,5184.000000,5184.000000,5184.000000,5184.000000
mean,23.741628,5.333237,10.855864,8.887731,18.264313,14.718962,8.603086
std,11.164247,3.076983,6.928117,5.650804,6.461616,5.121851,8.921401
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,15.600000,3.400000,5.800000,4.900000,14.400000,11.175000,4.200000
50%,22.500000,4.800000,9.400000,7.500000,17.800000,14.200000,6.400000
75%,30.000000,6.600000,14.300000,11.500000,21.400000,17.700000,9.600000
max,97.900000,56.200000,56.100000,48.400000,87.100000,56.500000,95.100000


## Step 3 — Prepare for Modeling

In [13]:
X = model_df_clean[features]
y = model_df_clean[target]

print(f"Features shape: {X.shape}")
print(f"Target distribution:")
print(y.value_counts())

Features shape: (5184, 7)
Target distribution:
risk_bucket
High SVI + Low ERR     2007
Low SVI + Low ERR      1818
High SVI + High ERR     833
Low SVI + High ERR      526
Name: count, dtype: int64


In [14]:
# Correlation matrix of features
X.corr().round(2)

,EP_POV150,EP_UNEMP,EP_NOHSDP,EP_UNINSUR,EP_AGE65,EP_DISABL,EP_NOVEH
EP_POV150,1.00,0.51,0.61,0.43,-0.16,0.49,0.41
EP_UNEMP,0.51,1.00,0.38,0.17,-0.15,0.25,0.32
EP_NOHSDP,0.61,0.38,1.00,0.58,-0.19,0.30,0.22
EP_UNINSUR,0.43,0.17,0.58,1.00,-0.21,0.14,-0.01
EP_AGE65,-0.16,-0.15,-0.19,-0.21,1.00,0.40,-0.15
EP_DISABL,0.49,0.25,0.30,0.14,0.40,1.00,0.10
EP_NOVEH,0.41,0.32,0.22,-0.01,-0.15,0.10,1.00


In [ ]:
["%pip install scikit-learn",
"from sklearn.model_selection import train_test_split",
"from sklearn.ensemble import RandomForestClassifier",
"from sklearn.metrics import classification_report",
"",
"X = model_df_clean[features]",
"y = model_df_clean[target]",
"",
"X_train, X_test, y_train, y_test = train_test_split(",
"    X, y, test_size=0.3, random_state=42, stratify=y",
")",
"",
"model = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)",
"",
"model.fit(X_train, y_train)",
"preds = model.predict(X_test)",
"",
"print(classification_report(y_test, preds))"]

Note: you may need to restart the kernel to use updated packages.
                     precision    recall  f1-score   support

High SVI + High ERR       0.00      0.00      0.00       250
 High SVI + Low ERR       0.65      0.91      0.76       602
 Low SVI + High ERR       0.00      0.00      0.00       173
  Low SVI + Low ERR       0.69      0.89      0.78       602

           accuracy                           0.67      1627
          macro avg       0.33      0.45      0.38      1627
       weighted avg       0.49      0.67      0.57      1627



/Users/roh/Documents/GitHub/readmissions-risk-pipeline/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/roh/Documents/GitHub/readmissions-risk-pipeline/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/roh/Documents/GitHub/readmissions-risk-pipeline/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter t

In [16]:
import pandas as pd

feat_imp = pd.Series(model.feature_importances_, index=features).sort_values(
    ascending=False
)
print(feat_imp)

EP_NOHSDP     0.311020
EP_POV150     0.254657
EP_UNEMP      0.146083
EP_UNINSUR    0.120676
EP_NOVEH      0.077040
EP_DISABL     0.051345
EP_AGE65      0.039179
dtype: float64
